# 💳 Credit Scoring Model using Machine Learning

in this notebook i built a credit scoring model that predicts whether a person is likely to default on a loan in the next 2 years, based on their financial history.

**Dataset:** Give Me Some Credit (Kaggle) — 150,000 real records  
**Models used:** Logistic Regression, Decision Tree, Random Forest  
**Platform:** Google Colab

---

### what i did step by step:
1. loaded and explored the dataset
2. handled missing values and outliers
3. did feature engineering (debt-to-income ratio, risk flags)
4. visualized the data to understand patterns
5. trained three models and compared them
6. evaluated using precision, recall, f1-score and roc-auc
7. picked the best model and predicted creditworthiness of a new person

---

### target variable:
- **0** = person did NOT default (creditworthy)
- **1** = person DID default (high risk)

this is a **classification** problem — we are predicting a category (default or not), not a number.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    ConfusionMatrixDisplay
)

import joblib
import warnings
warnings.filterwarnings('ignore')

print("libraries loaded.")

## 2. Load the Dataset

loading directly from github — no manual download needed.

In [ ]:
url = "https://raw.githubusercontent.com/pplonski/datasets-for-start/master/credit/data.csv"
df = pd.read_csv(url)

# drop the Id column — just a row number, not useful
df = df.drop(columns=['Id'])

print(f"dataset shape: {df.shape}")
df.head()

## 3. Understanding the Columns

let me rename columns to more readable names so it is easier to work with.

In [ ]:
# rename columns to simpler names
df = df.rename(columns={
    'SeriousDlqin2yrs'                    : 'defaulted',
    'RevolvingUtilizationOfUnsecuredLines' : 'credit_utilization',
    'age'                                  : 'age',
    'NumberOfTime30-59DaysPastDueNotWorse' : 'late_30_59_days',
    'DebtRatio'                            : 'debt_ratio',
    'MonthlyIncome'                        : 'monthly_income',
    'NumberOfOpenCreditLinesAndLoans'      : 'open_credit_lines',
    'NumberOfTimes90DaysLate'              : 'late_90_days',
    'NumberRealEstateLoansOrLines'         : 'real_estate_loans',
    'NumberOfTime60-89DaysPastDueNotWorse' : 'late_60_89_days',
    'NumberOfDependents'                   : 'dependents'
})

print("columns after renaming:")
print(df.columns.tolist())
print("\nbasic stats:")
df.describe()

## 4. Check Missing Values and Target Balance

In [ ]:
print("missing values per column:")
print(df.isnull().sum())

print("\ntarget variable distribution:")
print(df['defaulted'].value_counts())
print(f"\ndefault rate: {df['defaulted'].mean()*100:.2f}%")

In [ ]:
# visualize class imbalance
plt.figure(figsize=(6, 4))
counts = df['defaulted'].value_counts()
sns.barplot(x=['not defaulted (0)', 'defaulted (1)'], y=counts.values,
            palette=['steelblue', 'tomato'])
plt.title('class distribution — target variable')
plt.ylabel('number of people')
plt.tight_layout()
plt.show()

# note: dataset is heavily imbalanced — only ~6.7% defaulted
# this is normal in credit data and we will use stratify in train_test_split
# to make sure both classes are represented in train and test sets

## 5. Handle Missing Values and Outliers

In [ ]:
# fill missing monthly income with median
# median is better than mean here because income data is skewed (a few very high earners)
df['monthly_income'] = df['monthly_income'].fillna(df['monthly_income'].median())

# fill missing dependents with 0 — safest assumption
df['dependents'] = df['dependents'].fillna(0)

# cap outliers
# credit utilization should be between 0 and 1 (0% to 100%)
# anything above 1 is a data entry error
df['credit_utilization'] = df['credit_utilization'].clip(0, 1)

# debt ratio above 10 is extreme and likely an error
df['debt_ratio'] = df['debt_ratio'].clip(0, 10)

print("missing values after cleaning:")
print(df.isnull().sum())
print("\nall missing values handled!")

## 6. Feature Engineering

creating new features from existing ones that might help the model understand risk better.

In [ ]:
# total times late on any payment
df['total_late_payments'] = (
    df['late_30_59_days'] +
    df['late_60_89_days'] +
    df['late_90_days']
)

# monthly debt amount (debt ratio x income)
df['monthly_debt'] = df['debt_ratio'] * df['monthly_income']

# income per dependent — how much money available per person in household
# add 1 to avoid dividing by zero (the person themselves)
df['income_per_dependent'] = df['monthly_income'] / (df['dependents'] + 1)

# flag: has this person ever been 90+ days late?
# 1 = yes (high risk signal), 0 = no
df['ever_90_days_late'] = (df['late_90_days'] > 0).astype(int)

print("new features created:")
print("  total_late_payments, monthly_debt, income_per_dependent, ever_90_days_late")
print(f"\nnew dataset shape: {df.shape}")
df.head()

## 7. Exploratory Data Analysis (EDA)

In [ ]:
# age distribution by default status
plt.figure(figsize=(9, 4))
df[df['defaulted'] == 0]['age'].hist(bins=40, alpha=0.6,
                                      color='steelblue', label='not defaulted')
df[df['defaulted'] == 1]['age'].hist(bins=40, alpha=0.6,
                                      color='tomato', label='defaulted')
plt.title('age distribution by default status')
plt.xlabel('age')
plt.ylabel('count')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# credit utilization vs default
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x='defaulted', y='credit_utilization',
            palette=['steelblue', 'tomato'])
plt.title('credit utilization vs default status')
plt.xlabel('defaulted (0 = no, 1 = yes)')
plt.ylabel('credit utilization')
plt.tight_layout()
plt.show()

In [ ]:
# correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5)
plt.title('correlation matrix')
plt.tight_layout()
plt.show()

In [ ]:
# default rate by number of late payments
plt.figure(figsize=(9, 4))
late_default = df.groupby('total_late_payments')['defaulted'].mean().head(10)
sns.barplot(x=late_default.index, y=late_default.values, palette='Reds_r')
plt.title('default rate by total late payments')
plt.xlabel('total late payments')
plt.ylabel('default rate')
plt.tight_layout()
plt.show()

## 8. Prepare Data for Training

In [ ]:
# X = inputs, y = target (defaulted or not)
X = df.drop('defaulted', axis=1)
y = df['defaulted']

print(f"features used for training: {X.columns.tolist()}")
print(f"\ntotal features : {X.shape[1]}")
print(f"total samples  : {X.shape[0]}")

## 9. Train Test Split + Scaling

- **stratify=y** makes sure both classes (defaulted / not) are equally represented in train and test — important when data is imbalanced
- **StandardScaler** scales all features to the same range — important for logistic regression

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # keeps class ratio same in train and test
)

print(f"training samples : {len(X_train)}")
print(f"testing samples  : {len(X_test)}")

# scale features — logistic regression needs this
# decision tree and random forest do not need scaling but it does not hurt
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit on train, transform train
X_test_s  = scaler.transform(X_test)        # only transform test (never fit on test)

## 10. Train Three Models

- **logistic regression** — simple, fast, good baseline for classification
- **decision tree** — makes decisions using yes/no questions, easy to visualize
- **random forest** — 100 decision trees combined, usually the most accurate

In [ ]:
# logistic regression (uses scaled data)
lr_model = LogisticRegression(max_iter=2000, random_state=42)
lr_model.fit(X_train_s, y_train)
print("logistic regression trained.")

# decision tree (uses original data)
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
print("decision tree trained.")

# random forest (uses original data)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
print("random forest trained.")

## 11. Evaluate All Three Models

for credit scoring we care about:
- **precision** — of all people we predicted as defaulters, how many actually defaulted?
- **recall** — of all actual defaulters, how many did we catch?
- **f1-score** — balance between precision and recall
- **roc-auc** — overall ability to separate defaulters from non-defaulters (higher = better)

In [ ]:
def evaluate(model, name, X_test_input, use_scaled=False):
    X_input = X_test_s if use_scaled else X_test_input
    preds      = model.predict(X_input)
    probs      = model.predict_proba(X_input)[:, 1]
    auc        = roc_auc_score(y_test, probs)

    print(f"\n{'='*45}")
    print(f" {name}")
    print(f"{'='*45}")
    print(f" ROC-AUC score : {auc:.4f}")
    print("\n classification report:")
    print(classification_report(y_test, preds,
                                 target_names=['not defaulted', 'defaulted']))
    return preds, probs, auc

lr_preds, lr_probs, lr_auc = evaluate(lr_model, "Logistic Regression", X_test, use_scaled=True)
dt_preds, dt_probs, dt_auc = evaluate(dt_model, "Decision Tree",       X_test)
rf_preds, rf_probs, rf_auc = evaluate(rf_model, "Random Forest",       X_test)

## 12. ROC Curve — comparing all three models

In [ ]:
plt.figure(figsize=(8, 6))

for probs, name, auc in [
    (lr_probs, 'Logistic Regression', lr_auc),
    (dt_probs, 'Decision Tree',       dt_auc),
    (rf_probs, 'Random Forest',       rf_auc)
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

# diagonal line = random guessing (AUC = 0.5)
plt.plot([0, 1], [0, 1], 'k--', label='random guess (AUC = 0.5)')

plt.xlabel('false positive rate')
plt.ylabel('true positive rate')
plt.title('ROC curve — all models')
plt.legend()
plt.tight_layout()
plt.show()

## 13. Confusion Matrix — best model

In [ ]:
# pick best model based on ROC-AUC
scores     = {'Logistic Regression': lr_auc, 'Decision Tree': dt_auc, 'Random Forest': rf_auc}
best_name  = max(scores, key=scores.get)
best_preds = {'Logistic Regression': lr_preds, 'Decision Tree': dt_preds, 'Random Forest': rf_preds}[best_name]

print(f"best model: {best_name} (ROC-AUC = {scores[best_name]:.4f})")

# confusion matrix
# rows = actual,  columns = predicted
# top-left  = correctly predicted not defaulted
# top-right = predicted not defaulted but actually defaulted (missed!)
# bottom-left  = predicted defaulted but actually fine (false alarm)
# bottom-right = correctly predicted defaulted
cm = confusion_matrix(y_test, best_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['not defaulted', 'defaulted'])

fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title(f'confusion matrix — {best_name}')
plt.tight_layout()
plt.show()

## 14. Feature Importance

In [ ]:
importances = rf_model.feature_importances_
feature_names = X.columns.tolist()

imp_df = pd.DataFrame({
    'feature'   : feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

plt.figure(figsize=(9, 6))
sns.barplot(data=imp_df, x='importance', y='feature', palette='viridis')
plt.title('feature importance — random forest')
plt.xlabel('importance score')
plt.tight_layout()
plt.show()

print(imp_df.to_string(index=False))

## 15. Save the Best Model

In [ ]:
best_models = {
    'Logistic Regression': lr_model,
    'Decision Tree'      : dt_model,
    'Random Forest'      : rf_model
}

joblib.dump(best_models[best_name], '/content/credit_score_model.pkl')
joblib.dump(scaler,                 '/content/credit_scaler.pkl')

print(f"model saved: {best_name}")
print("files saved to /content/")

## 16. Predict Creditworthiness of a New Person

testing the model on someone new — not in the dataset.

In [ ]:
# example: a 35 year old with moderate debt and clean payment history
new_person = pd.DataFrame([{
    'credit_utilization'  : 0.35,
    'age'                 : 35,
    'late_30_59_days'     : 0,
    'debt_ratio'          : 0.4,
    'monthly_income'      : 5000,
    'open_credit_lines'   : 5,
    'late_90_days'        : 0,
    'real_estate_loans'   : 1,
    'late_60_89_days'     : 0,
    'dependents'          : 1,
    'total_late_payments' : 0,
    'monthly_debt'        : 2000,
    'income_per_dependent': 2500,
    'ever_90_days_late'   : 0
}])

# use best model to predict
model_to_use = best_models[best_name]

if best_name == 'Logistic Regression':
    new_person_input = scaler.transform(new_person)
else:
    new_person_input = new_person

prediction   = model_to_use.predict(new_person_input)[0]
probability  = model_to_use.predict_proba(new_person_input)[0][1]

print("--- new person profile ---")
print("age              : 35")
print("monthly income   : $5,000")
print("debt ratio       : 0.4")
print("credit usage     : 35%")
print("late payments    : 0")

print(f"\nprediction       : {'HIGH RISK — likely to default' if prediction == 1 else 'LOW RISK — creditworthy'}")
print(f"default probability : {probability*100:.1f}%")

---

## Results Summary

| model | ROC-AUC | notes |
|---|---|---|
| logistic regression | ~0.81 | simple, fast, good baseline |
| decision tree | ~0.85 | better at capturing complex patterns |
| random forest | ~0.84 | ensemble of trees, most stable |

---

### key findings:
- **credit utilization** and **total late payments** were the strongest predictors of default
- people who were ever 90+ days late had a significantly higher default rate
- younger people had slightly higher default rates in this dataset
- the dataset is heavily imbalanced (~93% non-default vs 7% default) which is normal in real credit data

### what roc-auc means:
- **0.5** = model is guessing randomly (useless)
- **0.8+** = good model
- **0.9+** = excellent model

all three models scored above 0.80 which is solid for a credit scoring problem.